# 11 — So sánh QAOA p=1 và p=2

> **English version below.**

Notebook 10 đã review đủ 30 lượt Windows/30.720 shots. Chuẩn hóa với khởi tạo chung giúp choice có 10 mẫu tối ưu/5 lượt thành công, nhưng reference chỉ có 3 mẫu/2 lượt; reference có một mẫu ground state tại seed 52. Chưa có cấu hình chung cải thiện đáng tin cậy cả hai input.

Bước này giữ **(A,B)=(3,3), H_execution=H_raw/kappa**, khởi tạo p=1=[0.3,1.0], và so với p=2=[0.3,0.0,1.0,0.0]. Hai mạch bắt đầu cùng trạng thái. Đây là protocol để kiểm tra độ sâu, không phải tuyên bố khởi tạo chung tốt nhất cho reference.

Hai độ sâu × năm seed mới 55–59 = **10 lượt mỗi instance, 20 lượt cả hai**. Cả hai dùng shots=1024/maxiter=80/rhobeg=1. Ngân sách maxiter tăng cho cả đối chứng p=1. Giữ từng input riêng, ghi cả các lượt hoàn tất có 0 nghiệm và không sửa bit.

## Bước 1 — Mở đúng repo và kernel

Chép đủ bảy tệp vào đúng đường dẫn, chọn Conda nt409-khiem/Python 3.12 rồi Restart Kernel. Module chạy QAOA chỉ bổ sung nguồn mã và thư mục kết quả mới; mô hình QUBO và backend giữ nguyên. Không cài dependency mới. Báo cáo cũ giữ nguyên; source hash mới yêu cầu manifest mới cho đợt này.

In [1]:
import gc
import json
import sys
from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd

ROOT = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
             if (p / "src/depth_campaign.py").is_file()
             and (p / "data/instances").is_dir()), None)
assert ROOT is not None, "Chép đủ file src và mở notebook bên trong repo."
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.qubo import build_qubo, energy_parts
from src.ising import build_ising, ising_energy, scale_ising, scaled_qaoa_point
from src.qaoa_experiment import (
    prepare_reference, probe_qaoa_resources, run_qaoa_experiment,
    save_experiment_report, experiment_environment,
)
from src.penalty_analysis import penalty_diagnostics
from src.depth_campaign import (
    embed_p1_start, make_design, make_trials, new_campaign, save_campaign, report_row, verify_campaign,
)
assert sys.version_info[:2] == (3, 12), "Chọn kernel Python 3.12 của dự án."
print("Repo:", ROOT)
print("Runtime:", experiment_environment()["runtime"])
print("Nguồn:", experiment_environment()["source_sha256"])

Repo: C:\Users\Khiem\quang-tum-com-pu-ting\nt409-task-placement
Runtime: {'python_version': '3.12.14', 'platform': 'Windows-11-10.0.26200-SP0', 'numpy_version': '2.5.3', 'scipy_version': '1.18.1', 'qiskit_version': '2.5.2', 'qiskit_optimization_version': '0.7.0', 'qiskit_aer_version': '0.17.2'}
Nguồn: {'placement.py': '7cfa397beb22fdf3feddedc0d889e104a619d6ef3e641ee72a62d82b087deb35', 'qubo.py': '7ac7d64500ce1fe73b1b0a34402d51eb88be33372c3906db4b7197552371e9ae', 'ising.py': 'c8a01c974d5976d96999324bdb871c625f4079a94a9daa5736378b331e9afc57', 'qaoa_experiment.py': '20a3f69a493bcd4a96843081e0553fe9b8d73d8158986b215458d9c19f474e70', 'penalty_analysis.py': '1ee33001b1ac723df6d2c9bb541ecac27221e30fa77c1dc9fb9777d3d87363ba', 'independent_campaign.py': 'af953f6f78b4fac9ff49522f3cb8f9677499eb3048401219bd7d411f7ac0c1eb', 'scaling_campaign.py': 'dce7f8c22d56b1cd7341b269b7ecb55ab9269d025e2ab0d8087f8c0c8ba5810f', 'depth_campaign.py': '443c1019e9cb70da3b2df3cd500520d31534489a32c62869323ca5c3c5129b74

## Bước 2 — Cố định thiết kế trước khi xem kết quả

Chạy choice trước. Lượt kế tiếp chọn reference sau khi lưu output và Restart Kernel. Hai độ sâu chạy ghép theo seed: p=1/seed55, p=2/seed55, rồi seed56…

Thứ tự Qiskit là [beta_0,…,beta_(p-1),gamma_0,…,gamma_(p-1)]. Vì vậy p=2 dùng [0.3,0,1,0], **không phải** [0.3,1,0,0]. Các góc thêm bằng 0 làm lớp cuối thành identity; chúng vẫn là biến được COBYLA điều chỉnh khi tối ưu.

Khởi tạo cố định này không lấy góc tối ưu từ một lượt p=1 trước đó. Mỗi lượt có cùng giới hạn 80 lần đánh giá/1024 shots mỗi lần; thời gian/cổng thực tế của p=2 có thể nhiều hơn. Đây là thăm dò nhiều seed trên hai fixture đã có, không phải kiểm tra khái quát sang input mới.

In [2]:
# STUDY_INSTANCE = "p1_choice"  # Sau đó đổi thành "p0_reference".
STUDY_INSTANCE = "p0_reference"
P1_START = [0.3,1.0]
SEEDS = [55,56,57,58,59]
BASE_CONFIG = {"shots":1024,"maxiter":80,"max_qubits":23,"max_seconds":300,
    "aer_threads":4,"aer_memory_mb":2048,"min_available_gib":2.0,
    "cobyla_rhobeg":1.0,"hamiltonian_scaling":"max_abs"}
RESUME_MANIFEST = None
assert STUDY_INSTANCE in ("p1_choice","p0_reference")
instance = json.loads((ROOT/"data/instances"/f"{STUDY_INSTANCE}.json").read_text(encoding="utf-8"))
bundle = build_qubo(instance,assignment_penalty=3,capacity_penalty=3)
reference = prepare_reference(bundle)
DESIGN = make_design(instance,bundle,SEEDS,BASE_CONFIG,P1_START)
planned = make_trials(DESIGN)
assert len(planned)==10 and len(SEEDS)==5 and not set(SEEDS)&set(range(42,55))
print("Instance:",STUDY_INSTANCE,"| penalties:",bundle["penalties"],"| kappa:",DESIGN["normalization"]["scale"])
print("Số lượt:",len(planned),"| cùng ngân sách:",BASE_CONFIG["maxiter"],"lần đánh giá")
display(pd.DataFrame([{"arm":t["arm_id"],"seed":t["seed"],"p":t["config"]["reps"],
    "maxiter":t["config"]["maxiter"],"shots":t["config"]["shots"],
    "initial_point_execution":t["config"]["initial_point"]} for t in planned]))

Instance: p0_reference | penalties: {'assignment': 3.0, 'capacity': 3.0} | kappa: 257.5
Số lượt: 10 | cùng ngân sách: 80 lần đánh giá


,arm,seed,p,maxiter,shots,initial_point_execution
0,p1_control,55,1,80,1024,"[0.3, 1.0]"
1,p2_embedded,55,2,80,1024,"[0.3, 0.0, 1.0, 0.0]"
2,p1_control,56,1,80,1024,"[0.3, 1.0]"
3,p2_embedded,56,2,80,1024,"[0.3, 0.0, 1.0, 0.0]"
4,p1_control,57,1,80,1024,"[0.3, 1.0]"
5,p2_embedded,57,2,80,1024,"[0.3, 0.0, 1.0, 0.0]"
6,p1_control,58,1,80,1024,"[0.3, 1.0]"
7,p2_embedded,58,2,80,1024,"[0.3, 0.0, 1.0, 0.0]"
8,p1_control,59,1,80,1024,"[0.3, 1.0]"
9,p2_embedded,59,2,80,1024,"[0.3, 0.0, 1.0, 0.0]"


## Bước 3 — Hiểu và kiểm chứng lớp QAOA thêm vào

U_C(gamma)=exp(-i gamma H_execution), U_M(beta)=exp(-i beta sum X). Một lớp gồm cost rồi mixer. p=2 có hai lớp và bốn góc. Khi beta_1=gamma_1=0, lớp mới là identity, nên p=2 chứa trạng thái p=1 như một trường hợp đặc biệt.

Do đó mức năng lượng kỳ vọng thấp nhất có thể biểu diễn ở p=2 không cao hơn p=1 nếu tìm được tối ưu toàn cục. **Điều này không bảo đảm COBYLA với ngân sách hữu hạn tìm được trạng thái đó, hoặc tỷ lệ placement tối ưu tăng.**

Kiểm chứng trạng thái trên bài 4 qubit để không cấp phát statevector 20/23 qubit cho phần kiểm chứng. Trên input thật, kiểm tra 10 vector năng lượng. Số qubit không đổi: choice=20/reference=23; số góc tăng 2→4.

In [3]:
from qiskit.circuit.library import QAOAAnsatz
from qiskit.quantum_info import Statevector

raw_operator,offset = build_ising(bundle)
scaled_operator,scaling = scale_ising(raw_operator,"max_abs")
assert scaling==DESIGN["normalization"] and reference["all_qubo_minimizers_feasible"]
rng = np.random.default_rng(409)
vectors = [[0]*raw_operator.num_qubits,reference["example_minimizer_bits"],
           *rng.integers(0,2,size=(8,raw_operator.num_qubits)).tolist()]
for bits in vectors:
    expected = energy_parts(bundle,bits)["energy"]
    assert np.isclose(expected,bundle["qubo"].objective.evaluate(bits),rtol=0,atol=1e-7)
    assert np.isclose(expected,ising_energy(raw_operator,offset,bits),rtol=0,atol=1e-7)
    assert np.isclose(expected,scaling["scale"]*ising_energy(scaled_operator,0,bits)+offset,rtol=0,atol=1e-7)
tiny = build_qubo({"schema_version":1,"instance_id":"embedding_check_only","objective":"min_active_hosts",
    "tasks":[{"id":"T0","cpu":1,"ram":1}],"hosts":[{"id":"H0","cpu":1,"ram":1}]},
    assignment_penalty=3,capacity_penalty=3)
tiny_raw,_ = build_ising(tiny)
tiny_scaled,_ = scale_ising(tiny_raw,"max_abs")
for point in (P1_START,[-0.4,2.1],[1.2,-0.7]):
    before = Statevector(QAOAAnsatz(cost_operator=tiny_scaled,reps=1).assign_parameters(point))
    after = Statevector(QAOAAnsatz(cost_operator=tiny_scaled,reps=2).assign_parameters(embed_p1_start(point)))
    assert before.equiv(after,rtol=0,atol=1e-10)
    assert np.isclose(before.expectation_value(tiny_scaled),after.expectation_value(tiny_scaled),rtol=0,atol=1e-10)
del before,after,tiny,tiny_raw,tiny_scaled
circuits = {p:QAOAAnsatz(cost_operator=scaled_operator,reps=p) for p in (1,2)}
for p,circuit in circuits.items():
    assert [str(v) for v in circuit.parameters]==[f"β[{i}]" for i in range(p)]+[f"γ[{i}]" for i in range(p)]
    assert circuit.num_qubits==raw_operator.num_qubits and circuit.num_parameters==2*p
print("PASS — 10 vector năng lượng, 3 trạng thái p=1→p=2 và thứ tự góc 2/4 tham số.")
display(pd.DataFrame([{"instance":STUDY_INSTANCE,"p":p,"qubits":c.num_qubits,"parameters":c.num_parameters,
    "parameter_names":[str(v) for v in c.parameters],"kappa":scaling["scale"],
    "original_optimum":reference["original_best_objective"],"qubo_minimum":reference["minimum_energy"]}
    for p,c in circuits.items()]))
del circuits

PASS — 10 vector năng lượng, 3 trạng thái p=1→p=2 và thứ tự góc 2/4 tham số.


c:\Users\Khiem\anaconda3\envs\nt409-khiem\Lib\site-packages\scipy\sparse\linalg\_dsolve\linsolve.py:653: SparseEfficiencyWarning: splu converted its input to CSC format
  return splu(A).solve
c:\Users\Khiem\anaconda3\envs\nt409-khiem\Lib\site-packages\scipy\sparse\linalg\_matfuncs.py:706: SparseEfficiencyWarning: spsolve is more efficient when sparse b is in the CSC matrix format
  return spsolve(Q, P)
c:\Users\Khiem\anaconda3\envs\nt409-khiem\Lib\site-packages\scipy\sparse\_index.py:174: SparseEfficiencyWarning: Changing the sparsity structure of a csr_matrix is expensive. lil and dok are more efficient.
  self._set_intXint(row, col, x.flat[0])


,instance,p,qubits,parameters,parameter_names,kappa,original_optimum,qubo_minimum
0,p0_reference,1,23,2,"[β[0], γ[0]]",257.5,2,2.0
1,p0_reference,2,23,4,"[β[0], β[1], γ[0], γ[1]]",257.5,2,2.0


## Bước 4 — Đo thử thời gian và RAM trước đợt chạy

Đo p=1/p=2 tại khởi tạo protocol, rồi đo thêm p=2 với [0.3,0.2,1.0,0.5] để lớp thứ hai hoạt động. Góc của lần đo thêm chỉ phục vụ đo tài nguyên, không đổi điểm khởi tạo tối ưu và không được tính là lượt benchmark.

Dùng thời gian lấy mẫu lớn hơn của hai lần p=2 để ước lượng ngân sách, cộng phần setup và hệ số dự phòng 1.5. Đây vẫn là ước lượng, không phải cam kết thời gian chạy. Giới hạn 300 giây kiểm tra giữa các lần đánh giá; Aer giới hạn bộ nhớ trạng thái, còn RSS là đỉnh vòng đời tiến trình.

In [4]:
probe_configs = {arm:next(t["config"] for t in planned if t["arm_id"]==arm) for arm in DESIGN["arms"]}
probe_configs["p2_active_layer_probe"] = {**probe_configs["p2_embedded"],"initial_point":[0.3,0.2,1.0,0.5]}
probes = {}
for name,config in probe_configs.items():
    print("Đo thử:",STUDY_INSTANCE,name,flush=True)
    probes[name] = probe_qaoa_resources(bundle,config)
display(pd.DataFrame([{"probe":name,"p":probe_configs[name]["reps"],
    "initial_mean_energy_raw":v["initial_sample_mean_qubo_energy"],"sampling_seconds":v["sampling_seconds"],
    "setup_seconds":v["setup_seconds"],"rough_sampling_seconds_per_run":v["rough_maxiter_plus_final_sampling_seconds"],
    "process_peak_rss_mib":v["resources"]["process_peak_rss_mib"]} for name,v in probes.items()]))
print("Chênh năng lượng hai khởi đầu tương đương (mẫu hữu hạn):",
    probes["p2_embedded"]["initial_sample_mean_qubo_energy"]-probes["p1_control"]["initial_sample_mean_qubo_energy"])
estimated = {}
for arm,names in {"p1_control":["p1_control"],"p2_embedded":["p2_embedded","p2_active_layer_probe"]}.items():
    estimated[arm] = max(probes[n]["setup_seconds"] for n in names)+1.5*max(
        probes[n]["rough_maxiter_plus_final_sampling_seconds"] for n in names)
print("Ước tính mỗi lượt có dự phòng (giây):",{k:round(v,2) for k,v in estimated.items()})
print("Ước tính lấy mẫu/setup cả ma trận (phút):",round(len(SEEDS)*sum(estimated.values())/60,2))
if max(estimated.values())>=BASE_CONFIG["max_seconds"]:
    raise RuntimeError("Đo thử chạm ngân sách: gửi bảng Bước 4 để điều chỉnh protocol trước Bước 6.")

Đo thử: p0_reference p1_control
Đo thử: p0_reference p2_embedded
Đo thử: p0_reference p2_active_layer_probe


,probe,p,initial_mean_energy_raw,sampling_seconds,setup_seconds,rough_sampling_seconds_per_run,process_peak_rss_mib
0,p1_control,1,1505.559570,0.554178,0.072088,44.888418,302.035156
1,p2_embedded,2,1505.559570,0.959994,0.059523,77.759554,303.425781
2,p2_active_layer_probe,2,1764.972656,1.003195,0.058174,81.258795,303.972656


Chênh năng lượng hai khởi đầu tương đương (mẫu hữu hạn): 0.0
Ước tính mỗi lượt có dự phòng (giây): {'p1_control': 67.4, 'p2_embedded': 121.95}
Ước tính lấy mẫu/setup cả ma trận (phút): 15.78


## Bước 5 — Tạo hoặc tiếp tục manifest riêng

Đợt mới dùng RESUME_MANIFEST=None. Nếu gián đoạn, dùng đúng đường dẫn manifest depth đã in, giữ nguyên input/design/code/runtime và chạy các cell từ đầu. Báo cáo hoàn tất được kiểm tra rồi bỏ qua; mỗi report được lưu ngay sau lượt chạy. Manifest notebook 10 không dùng để resume notebook 11.

In [5]:
if RESUME_MANIFEST is None:
    campaign = new_campaign(DESIGN)
    manifest_path = ROOT/"data/reference_results/phase4d_depth_runs"/f"depth_{STUDY_INSTANCE}_{campaign['campaign_id']}.json"
    save_campaign(manifest_path,campaign)
else:
    manifest_path = (ROOT/RESUME_MANIFEST).resolve()
    assert manifest_path.is_relative_to(ROOT),"Manifest phải nằm trong repo."
    campaign = json.loads(manifest_path.read_text(encoding="utf-8"))
    verify_campaign(ROOT,campaign,DESIGN,bundle,reference)
print("Manifest:",manifest_path.relative_to(ROOT).as_posix())
print("Hoàn tất / dự kiến:",campaign["completed_run_count"],"/",campaign["planned_run_count"])

Manifest: data/reference_results/phase4d_depth_runs/depth_p0_reference_20261007T070412286634Z.json
Hoàn tất / dự kiến: 0 / 10


## Bước 6 — Chạy hai độ sâu theo từng seed

Mỗi lượt tối ưu/lấy mẫu độc lập trong cùng giới hạn ngân sách. p=2 bắt đầu từ góc cố định đã ghi, không nhận nghiệm tối ưu vừa chạy của p=1. Lượt hoàn tất có 0 nghiệm vẫn được lưu. Nếu bị ngắt/chạm thời gian, giữ các lượt đã hoàn tất, lượt chưa hoàn tất không tạo JSON kết quả.

Không thay BASE_CONFIG, SEEDS hoặc input giữa đợt: dựng design mới hoặc resume đúng design đã lưu. Thời gian của p=2 có thể dài hơn dù cùng số lần đánh giá hàm.

In [6]:
assert STUDY_INSTANCE == DESIGN["instance_id"],"Đổi instance: Restart Kernel và chạy lại từ Bước 1."
assert BASE_CONFIG == DESIGN["base_config"] and SEEDS == DESIGN["seeds"],"Cấu hình đổi: dựng lại design. "
assert planned == make_trials(DESIGN),"Ma trận lượt chạy đã đổi."
verify_campaign(ROOT,campaign,DESIGN,bundle,reference)
completed = {row["run_key"] for row in campaign["completed_runs"]}
for trial in planned:
    if trial["run_key"] in completed:
        print("Bỏ qua lượt đã hoàn tất:",trial["arm_id"],trial["seed"])
        continue
    assert experiment_environment() == DESIGN["environment"],"Mã/runtime đã đổi; tạo đợt mới."
    print("START",STUDY_INSTANCE,trial["arm_id"],"seed",trial["seed"],flush=True)
    report = run_qaoa_experiment(bundle,trial["config"],reference)
    if report["status"]!="completed":
        print("Dừng đợt:",report["stop_reason"],"Lượt chưa hoàn tất không được lưu.")
        del report
        gc.collect()
        break
    assert report["environment"] == DESIGN["environment"]
    report["scope"] = campaign["scope"]
    report["study"] = DESIGN["study"].copy()
    report["experiment"] = {"campaign_id":campaign["campaign_id"],"design_hash":campaign["design_hash"],
        "run_key":trial["run_key"],"arm_id":trial["arm_id"],"purpose":"matched_budget_depth_comparison"}
    started = perf_counter()
    report["penalty_analysis"] = penalty_diagnostics(bundle,report)
    report["penalty_analysis_seconds"] = perf_counter()-started
    path = save_experiment_report(ROOT,report)
    row = report_row(ROOT,path,report,trial)
    campaign["completed_runs"].append(row)
    save_campaign(manifest_path,campaign)
    completed.add(trial["run_key"])
    print("DONE | hợp lệ:",round(100*row["feasible_probability"],4),"% | tối ưu:",
          round(100*row["optimal_placement_probability"],4),"% | host tốt nhất:",row["observed_best_objective"],
          "| QAOA:",round(row["elapsed_seconds"],2),"s",flush=True)
    del report
    gc.collect()

START p0_reference p1_control seed 55
  evaluation 1: mean E=1505.559570, elapsed=0.6s
  evaluation 5: mean E=1075.033203, elapsed=2.8s
  evaluation 10: mean E=462.944336, elapsed=5.6s
  evaluation 15: mean E=502.647461, elapsed=8.3s
  evaluation 20: mean E=489.333984, elapsed=12.0s
  evaluation 25: mean E=513.450195, elapsed=14.9s
  evaluation 30: mean E=499.965820, elapsed=17.8s
DONE | hợp lệ: 0 % | tối ưu: 0 % | host tốt nhất: None | QAOA: 18.9 s
START p0_reference p2_embedded seed 55
  evaluation 1: mean E=1505.559570, elapsed=1.5s
  evaluation 5: mean E=1079.557617, elapsed=7.0s
  evaluation 10: mean E=498.010742, elapsed=13.7s
  evaluation 15: mean E=207.201172, elapsed=20.5s
  evaluation 20: mean E=210.119141, elapsed=27.4s
  evaluation 25: mean E=206.885742, elapsed=34.2s
  evaluation 30: mean E=193.525391, elapsed=40.6s
  evaluation 35: mean E=204.005859, elapsed=47.4s
  evaluation 40: mean E=199.399414, elapsed=54.2s
DONE | hợp lệ: 0.0977 % | tối ưu: 0.0977 % | host tốt nhất:

## Bước 7 — Xem từng lượt trước khi gộp

Ba cột probability đã đổi sang phần trăm. found_optimal cho biết lượt đó có ít nhất một placement tối ưu; số mẫu tối ưu là chỉ số khác. observed_best_objective/gap thiếu nghĩa là chưa thấy nghiệm hợp lệ, không thay bằng 0. dominant_bitstring_count>1 nghĩa có đồng hạng tần suất; None ở unique_mode_valid nghĩa không có mode duy nhất.

In [7]:
results = pd.DataFrame(campaign["completed_runs"])
if results.empty:
    print("Chưa có lượt hoàn tất.")
else:
    table = results[["arm_id","reps","seed","found_valid","found_optimal","feasible_probability",
        "optimal_placement_probability","qubo_ground_state_probability","observed_best_objective",
        "optimality_gap","dominant_bitstring_count","unique_mode_valid","cost_function_evals","elapsed_seconds"]].copy()
    for col in ("feasible_probability","optimal_placement_probability","qubo_ground_state_probability"):
        table[col] = 100*pd.to_numeric(table[col])
    display(table.round(6))
    print("Ba cột probability trong bảng đã đổi sang phần trăm.")

,arm_id,reps,seed,found_valid,found_optimal,feasible_probability,optimal_placement_probability,qubo_ground_state_probability,observed_best_objective,optimality_gap,dominant_bitstring_count,unique_mode_valid,cost_function_evals,elapsed_seconds
0,p1_control,1,55,False,False,0.000000,0.000000,0,NaN,NaN,5,None,31,18.901990
1,p2_embedded,2,55,True,True,0.097656,0.097656,0,2.0,0.0,1024,None,42,58.347257
2,p1_control,1,56,False,False,0.000000,0.000000,0,NaN,NaN,8,None,40,33.841721
3,p2_embedded,2,56,False,False,0.000000,0.000000,0,NaN,NaN,1024,None,39,53.212072
4,p1_control,1,57,True,True,0.195312,0.195312,0,2.0,0.0,9,None,28,23.217548
5,p2_embedded,2,57,True,True,0.195312,0.195312,0,2.0,0.0,1024,None,36,50.029506
6,p1_control,1,58,True,True,0.292969,0.292969,0,2.0,0.0,1024,None,49,35.189610
7,p2_embedded,2,58,True,True,0.097656,0.097656,0,2.0,0.0,1,False,41,59.253948
8,p1_control,1,59,True,True,0.097656,0.097656,0,2.0,0.0,9,None,34,26.677771
9,p2_embedded,2,59,False,False,0.000000,0.000000,0,NaN,NaN,2,None,38,48.583569


Ba cột probability trong bảng đã đổi sang phần trăm.


## Bước 8 — Chất lượng, chi phí và so sánh ghép seed

Chỉ số chính: optimal_pct_mean, đọc cùng optimal_runs và optimal_shots/5120. So riêng từng input. Bảng chi phí cho biết số lần đánh giá thực tế và thời gian; cùng maxiter không bắt buộc cùng số lần thực tế nếu optimizer dừng sớm.

Delta là p=2 trừ p=1: delta_optimal_pct_points dương nghĩa nhiều mẫu placement tối ưu hơn; delta_mean_energy âm nghĩa năng lượng giảm. Hai dấu này không bắt buộc cùng tốt. Std là độ lệch chuẩn giữa seed, không phải khoảng tin cậy. Năm seed và số mẫu tối ưu hiếm chỉ hỗ trợ đánh giá thăm dò.

Các lượt p=1/maxiter80 mới là đối chứng cho p=2/maxiter80. Số liệu notebook 10/maxiter35 là mốc lịch sử khác seed/ngân sách, không phải đối chứng trực tiếp về độ sâu.

In [8]:
summary = pd.DataFrame()
costs = pd.DataFrame()
paired = pd.DataFrame()
if not results.empty:
    shots = BASE_CONFIG["shots"]
    summary = results.groupby("arm_id",sort=False).agg(
        completed_runs=("run_key","size"),valid_runs=("found_valid","sum"),optimal_runs=("found_optimal","sum"),
        valid_shots=("feasible_probability",lambda s:int(round(shots*s.sum()))),
        optimal_shots=("optimal_placement_probability",lambda s:int(round(shots*s.sum()))),
        optimal_pct_mean=("optimal_placement_probability",lambda s:100*s.mean()),
        optimal_pct_std=("optimal_placement_probability",lambda s:100*s.std(ddof=1)),
        feasible_pct_mean=("feasible_probability",lambda s:100*s.mean()),
        ground_shots=("qubo_ground_state_probability",lambda s:int(round(shots*s.sum()))),
        mean_energy=("final_sample_mean_qubo_energy","mean"),
    ).reset_index()
    costs = results.groupby("arm_id",sort=False).agg(
        mean_evaluations=("cost_function_evals","mean"),mean_solver_seconds=("elapsed_seconds","mean"),
        total_solver_seconds=("elapsed_seconds","sum"),mean_postprocessing_seconds=("postprocessing_seconds","mean"),
        mean_diagnostic_seconds=("penalty_analysis_seconds","mean"),
        max_recorded_peak_rss_mib=("process_peak_rss_mib","max"),
    ).reset_index()
    display(summary.round(6))
    display(costs.round(6))
    control = results[results["arm_id"]=="p1_control"].set_index("seed")
    candidate = results[results["arm_id"]=="p2_embedded"].set_index("seed")
    paired = pd.DataFrame([{"seed":seed,
        "p1_optimal_shots":int(round(shots*control.loc[seed,"optimal_placement_probability"])),
        "p2_optimal_shots":int(round(shots*candidate.loc[seed,"optimal_placement_probability"])),
        "delta_optimal_pct_points":100*(candidate.loc[seed,"optimal_placement_probability"]-control.loc[seed,"optimal_placement_probability"]),
        "delta_feasible_pct_points":100*(candidate.loc[seed,"feasible_probability"]-control.loc[seed,"feasible_probability"]),
        "delta_mean_energy":candidate.loc[seed,"final_sample_mean_qubo_energy"]-control.loc[seed,"final_sample_mean_qubo_energy"],
        "p1_evaluations":int(control.loc[seed,"cost_function_evals"]),"p2_evaluations":int(candidate.loc[seed,"cost_function_evals"]),
        "solver_seconds_ratio":candidate.loc[seed,"elapsed_seconds"]/control.loc[seed,"elapsed_seconds"]}
        for seed in sorted(set(candidate.index)&set(control.index))])
    display(paired.round(6))
    if not paired.empty:
        delta = paired["delta_optimal_pct_points"]
        print("p=2 so với p=1 | hơn/bằng/kém:",int((delta>0).sum()),int((delta==0).sum()),int((delta<0).sum()))
    print("Số cặp seed đầy đủ:",len(paired),"/",len(SEEDS))
    print("Baseline placement tối ưu đều (%):",100*reference["uniform_baseline"]["optimal_placement_probability"])

,arm_id,completed_runs,valid_runs,optimal_runs,valid_shots,optimal_shots,optimal_pct_mean,optimal_pct_std,feasible_pct_mean,ground_shots,mean_energy
0,p1_control,5,3,3,6,6,0.117188,0.127328,0.117188,0,350.897852
1,p2_embedded,5,3,3,4,4,0.078125,0.081705,0.078125,0,188.509180


,arm_id,mean_evaluations,mean_solver_seconds,total_solver_seconds,mean_postprocessing_seconds,mean_diagnostic_seconds,max_recorded_peak_rss_mib
0,p1_control,36.4,27.565728,137.828640,2.111577,0.235870,311.90625
1,p2_embedded,39.2,53.885271,269.426353,1.984256,0.203572,311.90625


,seed,p1_optimal_shots,p2_optimal_shots,delta_optimal_pct_points,delta_feasible_pct_points,delta_mean_energy,p1_evaluations,p2_evaluations,solver_seconds_ratio
0,55,0,1,0.097656,0.097656,-270.481445,31,42,3.086832
1,56,0,0,0.000000,0.000000,-116.602539,40,39,1.572381
2,57,2,2,0.000000,0.000000,-221.582031,28,36,2.154814
3,58,3,1,-0.195312,-0.195312,-0.814453,49,41,1.683848
4,59,1,0,-0.097656,-0.097656,-202.462891,34,38,1.821125


p=2 so với p=1 | hơn/bằng/kém: 1 2 2
Số cặp seed đầy đủ: 5 / 5
Baseline placement tối ưu đều (%): 0.09765625


## Bước 9 — Đọc hai report thật của cùng seed

Mặc định xem seed55. So góc theo tên, năng lượng trung bình và best_observed_valid của p=1/p=2. Mẫu hợp lệ tốt nhất là hậu xử lý trên mẫu đo; tần suất đứng đầu và các đồng hạng phải công bố riêng khi đối chiếu quy tắc chọn đầu ra của proposal. Placement tối ưu không yêu cầu slack đúng; ground state toàn QUBO có yêu cầu mạnh hơn.

In [9]:
INSPECT_SEED = SEEDS[0]
for entry in campaign["completed_runs"]:
    if entry["seed"]!=INSPECT_SEED:
        continue
    inspected = json.loads((ROOT/entry["report"]).read_text(encoding="utf-8"))
    print("Nguồn:",entry["report"])
    print("Góc execution theo tên:",dict(zip(inspected["parameter_names"],inspected["optimal_point"])))
    print("Góc theo H gốc:",dict(zip(inspected["parameter_names"],inspected["optimal_point_raw_hamiltonian"])))
    print("Năng lượng trung bình QUBO:",inspected["metrics"]["final_sample_mean_qubo_energy"])
    print("Nghiệm hợp lệ tốt nhất:",inspected["metrics"]["best_observed_valid"])
    print("Chẩn đoán:",inspected["penalty_analysis"])
    display(pd.DataFrame([{"probability":s["probability"],"energy_raw":s["energy"],
        "feasible":s["decoded"]["feasible"],"hosts":s["decoded"]["objective"],
        "x":s["decoded"]["x"],"y":s["decoded"]["y"]} for s in inspected["final_samples"][:5]]))
    del inspected
    gc.collect()

Nguồn: data/reference_results/phase4d_depth_runs/p0_reference_p1_control_p1_seed55_20261007T070440251854Z.json
Góc execution theo tên: {'β[0]': 2.6019569222581405, 'γ[0]': 2.537599057874344}
Góc theo H gốc: {'β[0]': 2.6019569222581405, 'γ[0]': 0.009854753622812986}
Năng lượng trung bình QUBO: 462.9443359375
Nghiệm hợp lệ tốt nhất: None
Chẩn đoán: {'mean_active_hosts': 1.0234375, 'mean_assignment_error': 2.1328125, 'mean_capacity_equation_error': 151.8408203125, 'mean_total_squared_error': 153.9736328125, 'reconstructed_mean_energy': 462.9443359375, 'dominant_bitstring_count': 5, 'dominant_valid_bitstring_count': 0, 'unique_mode_valid': None, 'one_hot_probability': 0.0732421875}


,probability,energy_raw,feasible,hosts,x,y
0,0.001953,182.0,False,None,"[[0, 0], [0, 1], [0, 0], [0, 0]]","[1, 1]"
1,0.001953,738.0,False,None,"[[0, 1], [0, 1], [0, 1], [1, 0]]","[0, 0]"
2,0.001953,433.0,False,None,"[[0, 0], [0, 0], [0, 0], [1, 0]]","[0, 1]"
3,0.001953,357.0,False,None,"[[0, 0], [0, 0], [0, 0], [1, 1]]","[0, 0]"
4,0.001953,378.0,False,None,"[[0, 0], [0, 0], [0, 0], [1, 1]]","[0, 0]"


Nguồn: data/reference_results/phase4d_depth_runs/p0_reference_p2_embedded_p2_seed55_20261007T070540943885Z.json
Góc execution theo tên: {'β[0]': 1.5057239561316718, 'β[1]': 1.2065146731305374, 'γ[0]': 0.9397124176875167, 'γ[1]': -0.12213533169393048}
Góc theo H gốc: {'β[0]': 1.5057239561316718, 'β[1]': 1.2065146731305374, 'γ[0]': 0.0036493686123787055, 'γ[1]': -0.00047431196774341936}
Năng lượng trung bình QUBO: 192.462890625
Nghiệm hợp lệ tốt nhất: {'bits': [1, 0, 1, 0, 0, 1, 0, 1, 1, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0], 'energy': 8.0, 'probability': 0.0009765625, 'feasible': True, 'objective': 2, 'one_hot_ok': True, 'capacity_ok': True, 'x': [[1, 0], [1, 0], [0, 1], [0, 1]], 'y': [1, 1], 'usage': [[5, 6], [5, 8]]}
Chẩn đoán: {'mean_active_hosts': 1.48828125, 'mean_assignment_error': 2.5263671875, 'mean_capacity_equation_error': 61.1318359375, 'mean_total_squared_error': 63.658203125, 'reconstructed_mean_energy': 192.462890625, 'dominant_bitstring_count': 1024, 'dominant_valid_b

,probability,energy_raw,feasible,hosts,x,y
0,0.000977,137.0,False,None,"[[0, 0], [1, 0], [0, 0], [0, 0]]","[1, 1]"
1,0.000977,151.0,False,None,"[[0, 1], [0, 0], [0, 1], [0, 1]]","[0, 1]"
2,0.000977,311.0,False,None,"[[0, 0], [0, 0], [1, 0], [0, 0]]","[1, 1]"
3,0.000977,50.0,False,None,"[[0, 1], [0, 1], [0, 0], [0, 0]]","[1, 1]"
4,0.000977,50.0,False,None,"[[1, 1], [1, 1], [0, 0], [0, 0]]","[1, 1]"


## Bước 10 — Kiểm tra tính nhất quán và đủ lượt

PASS xác minh tính đúng của tệp, không tuyên bố p=2 thắng. Mốc hoàn tất mỗi instance: 10/10, không còn lượt, hai nhóm năm lượt và năm cặp seed. Nếu ít hơn, đợt mới hoàn tất một phần; resume đúng manifest. Sau choice, lưu bản có output rồi Restart Kernel để chạy reference.

In [10]:
saved = json.loads(manifest_path.read_text(encoding="utf-8"))
verified = verify_campaign(ROOT,saved,DESIGN,bundle,reference)
assert verified==campaign["completed_runs"]
if saved["completed_run_count"]==len(planned):
    assert summary["completed_runs"].tolist()==[5,5] and len(paired)==5
    print("PASS — đủ lượt, góc p=1/2, nguồn mã, năng lượng và mẫu đo đều khớp.")
else:
    print("PASS tính nhất quán; đợt CHƯA đủ lượt, dùng manifest này để resume.")
print("Instance:",STUDY_INSTANCE,"| hoàn tất / dự kiến:",saved["completed_run_count"],"/",saved["planned_run_count"])
print("Lượt còn lại:",len(saved["remaining_run_keys"]))
print("Gửi notebook có output, manifest",manifest_path.name,"và",saved["completed_run_count"],"JSON nguồn được tham chiếu.")
print("Thư mục:",manifest_path.parent.relative_to(ROOT).as_posix())

PASS — đủ lượt, góc p=1/2, nguồn mã, năng lượng và mẫu đo đều khớp.
Instance: p0_reference | hoàn tất / dự kiến: 10 / 10
Lượt còn lại: 0
Gửi notebook có output, manifest depth_p0_reference_20261007T070412286634Z.json và 10 JSON nguồn được tham chiếu.
Thư mục: data/reference_results/phase4d_depth_runs


## Sau khi chạy

Lưu choice thành 11_qaoa_depth_comparison_choice.ipynb. Restart Kernel, đặt STUDY_INSTANCE="p0_reference", RESUME_MANIFEST=None, chạy lại từ Bước 1 và lưu 11_qaoa_depth_comparison_reference.ipynb. Gửi hai notebook, hai manifest depth và 20 report nguồn (10 mỗi instance).

Mục tiêu review: p=2 có tăng tỷ lệ/lượt tìm được placement tối ưu với chi phí chấp nhận được hay không. Khởi tạo zero-layer có thể khó cho optimizer khi thêm góc; nếu không cải thiện, đó là kết quả của protocol này, không chứng minh mọi QAOA p=2 kém p=1. Review xong mới quyết định p=3 hoặc điều chỉnh optimizer/khởi tạo theo một đợt được ghi riêng. B/C tiếp tục baseline và phân tích; chưa khép phần so sánh cổ điển hoặc quy mô lớn của proposal.

---

## English version

Notebook 10's thirty Windows trials (30,720 shots) are fully reviewed. Shared normalized initialization improved choice descriptively (ten optimal shots, five successful trials), but reference produced three shots/two successful trials and one full-QUBO ground-state sample. A universally better shared configuration is not established.

This exploratory depth protocol fixes the unchanged input/QUBO at (assignment,capacity)=(3,3), executes H_raw/kappa, and compares p=1 [beta0,gamma0]=[0.3,1.0] with p=2 [beta0,beta1,gamma0,gamma1]=[0.3,0,1.0,0]. The appended cost/mixer layer is identity, so both start from the same physical state. These fixed starting angles do not reuse a learned p=1 optimum. Both arms receive the same cap of 80 function evaluations and 1024 shots per evaluation, COBYLA rhobeg 1, CPU Aer/four threads, 2048 MiB state memory, 2 GiB minimum available RAM and 300 seconds checked between evaluations. Five fresh seeds 55–59 are paired in execution order. Ten trials per fixture, twenty total; actual evaluations and wall-clock costs may differ.

The p=2 state family contains p=1 through the zero final layer, so its globally attainable minimum expected energy cannot be worse. Finite-budget COBYLA and optimal-placement probabilities have no such improvement guarantee; zero-layer initialization itself may be difficult to escape. Qubit counts remain 20/23 while angle counts rise from two to four. Quality is assessed separately for each input with success counts, optimal/feasible probabilities and shots, full-QUBO ground states, mean energy and costs. Ordinary expected energy is not optimal-placement probability. Standard deviation across five seeds is not a confidence interval. The same-budget p=1 arm is the depth control; notebook 10's different seeds/maxiter35 are historical evidence rather than a matched depth comparison.

Copy all seven package files into matching paths and restart the Conda Python 3.12 kernel. Execute steps 1–10: environment; fixed design; input energy/tiny-state/parameter-order checks; resource probes at both starts and an active p=2 diagnostic point; a new/resumed manifest; paired sequential optimization and per-run saving; individual trials; quality/cost/paired-seed tables; source reports; consistency and completeness. The active-layer point [0.3,0.2,1.0,0.5] is only a resource diagnostic, not another experiment arm or optimizer start. A 1.5-margin timing estimate is not a runtime guarantee. State checks use a four-qubit model without large validation statevectors.

New files live under data/reference_results/phase4d_depth_runs with depth_<instance>_<campaign_id>.json manifests. Resume only the same design/input/code/runtime; completed reports are verified and skipped. Completed zero-solution trials remain in all denominators, measured bits are not repaired, and best-valid postselection is separate from most-frequent/tied output rules. Raw QUBO energies and raw/execution gamma conversions remain explicit. Adding runner/depth-source provenance changes source hashes and rejects old-campaign resume; completed historical reports remain readable.

Run choice first, expect 10/10 with two five-trial groups and five paired rows, and save an executed copy. Restart, select reference with RESUME_MANIFEST=None, repeat and save. Submit both notebooks, two depth manifests and twenty referenced reports. Review before choosing p=3 or a separately documented optimization/initialization study; this step does not close the proposal's classical comparison or larger-instance scope.